# Partitioning [partitionBy()] and Bucketing [bucketBy()]

Third notebook in the PySpark learning series — using the same UC-volume dataset setup.

   
# Setup — Paths & Imports

Shared constants used across this notebook:

* `setup_data_dirs(dir_name)` — reusable helper: copies `data/{dir_name}` from the repo into a UC volume and creates its matching output volume (`{dir_name}` → `/Volumes/workspace/default/{dir_name}/`, outputs → `/Volumes/workspace/default/{dir_name}-output/`)
* `FLIGHT_DATA_DIR` / `OUTPUT_DIR` — dataset and write-output paths, derived from the helper
* `pprint` — pretty-printer for readable `dtypes` and schema output

In [0]:
import shutil
from pathlib import Path

from pprint import pprint

# Reusable setup — Unit Catalog (UC) volumes are the only workspace locations Spark can both
# READ from and WRITE to. This helper copies a dataset directory from the git
# repo into a UC volume and prepares its matching output directory.
REPO_DATA_ROOT = Path("/Workspace/Repos/mayurkedari16@gmail.com/pyspark-databricks/data")
VOLUME_ROOT = Path("/Volumes/workspace/default")


def setup_data_dirs(
    dir_name: str,
    data_volume: str | None = None,
    output_volume: str | None = None,
) -> tuple[str, str]:
    """
    Copy <repo>/data/{dir_name} into a UC volume and create its output directory.

    * data:   {REPO_DATA_ROOT}/{dir_name}  ->  {VOLUME_ROOT}/{data_volume or dir_name}
    * output:                                 {VOLUME_ROOT}/{output_volume or {dir_name}-output}

    Creates the volumes if they don't exist (CREATE VOLUME IF NOT EXISTS) and is
    idempotent — re-running safely overwrites the staged copies.

    Returns (data_dir, output_dir) as strings; data_dir ends in "/"
    for path concatenation (FLIGHT_DATA_DIR + "csv/*.csv").
    """
    # dir_name = Path(dir_name).resolve().as_posix()
    src = Path(REPO_DATA_ROOT) / dir_name
    if not src.is_dir():
        raise FileNotFoundError(f"No dataset directory at {src}")

    data_volume = data_volume or dir_name
    output_volume = output_volume or (dir_name if dir_name.endswith("-output") else f"{dir_name}-output")

    # A volume must exist in Unity Catalog before its /Volumes path is usable
    spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.default.`{data_volume}`")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.default.`{output_volume}`")

    data_dir = Path(VOLUME_ROOT) / data_volume
    output_dir = Path(VOLUME_ROOT) / output_volume

    shutil.copytree(src, data_dir, dirs_exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    return data_dir.as_posix() + "/", output_dir.as_posix() + "/"

In [0]:
# Wire up this notebook's dataset using the reusable helper — copies
# data/flight-data from the repo into the `write_ops_data` UC volume
# (idempotent) and points WRITE_OPS_OUTPUT_DIR at the `write_ops_data_output`
# volume, where every partition/bucket output in this notebook is written.
WRITE_OPS_DATA_DIR, WRITE_OPS_OUTPUT_DIR = setup_data_dirs(
    "flight-data",
    data_volume="write_ops_data",
    output_volume="write_ops_data_output",
)

print("Dataset Dir:", WRITE_OPS_DATA_DIR)
print("Outputs Dir:", WRITE_OPS_OUTPUT_DIR)

In [0]:
%sh

ls -l //Volumes/workspace/default/write_ops_data_output/csv_out/

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, 
    StructField, 
    IntegerType, 
    StringType, 
    DoubleType,
)


spark = SparkSession.builder.appName("write-operations").getOrCreate()

data = [
        # ("id", "name", "age", "salary", "country", "gender"),
        (1, "Manish", 26, 75000, "INDIA", "m"),
        (2, "Nikita", 23, 100000, "USA", "f"),
        (3, "Pritam", 22, 150000, "INDIA", "m"),
        (4, "Prantosh", 17, 200000, "JAPAN", "m"),
        (5, "Vikash", 31, 300000, "USA", "m"),
        (6, "Rahul", 55, 300000, "INDIA", "m"),
        (7, "Raju", 67, 540000, "USA", "m"),
        (8, "Praveen", 28, 70000, "JAPAN", "m"),
        (9, "Dev", 32, 150000, "JAPAN", "m"),
        (10, "Sherin", 16, 25000, "RUSSIA", "f"),
        (11, "Ragu", 12, 35000, "INDIA", "f"),
        (12, "Sweta", 43, 200000, "INDIA", "f"),
        (13, "Raushan", 48, 650000, "USA", "m"),
        (14, "Mukesh", 36, 95000, "RUSSIA", "m"),
        (15, "Prakash", 52, 750000, "INDIA", "m"),
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", DoubleType(), True),
    StructField("country", StringType(), True),
    StructField("gender", StringType(), True),
])
df = spark.createDataFrame(data, schema=schema, )
df.show(5)


# Partitioning — `partitionBy()`

Splits the output into separate subdirectories, one per distinct value of the partition column(s). Spark then **prunes** partitions at read time — queries filtering on the partition column skip irrelevant directories entirely.

[!NOTE]
Use **partitioning** when a column has a *low-to-moderate cardinality* and lends itself to logical grouping (e.g. `country`, `gender`, `year`).

[!WARNING]
If cardinality is too high, partitioning creates a flood of tiny files — one directory per value. In that case prefer **bucketing** instead (see below).

### When to partition vs. bucket

| Aspect | `partitionBy()` | `bucketBy()` |
| --- | --- | --- |
| Column cardinality | Low–moderate (e.g. country, gender) | High (e.g. id, user_id) |
| Layout on disk | Nested `key=value/` directories | Fixed number of bucket files |
| Pruning | Directory pruning at read time | Hash-based shuffle avoidance on joins |
| Best for | Filters / `WHERE` on the column | Joins / `GROUP BY` on the column |

In [0]:
from pathlib import Path
PARITION_BY_DIR = (Path(WRITE_OPS_OUTPUT_DIR) / "parition_by_data" ).resolve().as_posix()
print(f"{PARITION_BY_DIR = }")

# Single-column partitionBy("country") — low cardinality (4 distinct
# countries), so Spark writes one `country=<value>/` subdirectory per value.
# A later `WHERE country = 'INDIA'` read prunes every other directory.
(
    df.write.format("csv")
            .mode("overwrite")
            .option("header", True)
            .partitionBy("country")
            .save(PARITION_BY_DIR)
)


In [0]:
%sh

ls -ltr /Volumes/workspace/default/write_ops_data_output/parition_by_data/*

In [0]:
from pathlib import Path
PARITION_BY_DIR = (Path(WRITE_OPS_OUTPUT_DIR) / "parition_by_multi_cols_data" ).resolve().as_posix()
print(f"{PARITION_BY_DIR = }")

# Multi-column partitionBy("country", "salary") — nests `country=/salary=`
# directories. salary has many distinct values, so the partition count
# explodes (country × salary combos) — a reminder that the SECOND partition
# column should also be low-cardinality.
(
    df.write.format("csv")
            .mode("overwrite")
            .option("header", True)
            .partitionBy("country", "salary")
            .save(PARITION_BY_DIR)
)


In [0]:
%sh
ls -ltr /Volumes/workspace/default/write_ops_data_output/parition_by_multi_cols_data/country=INDIA/**

In [0]:
from pathlib import Path
PARITION_BY_DIR = (Path(WRITE_OPS_OUTPUT_DIR) / "parition_by_country_gender" ).resolve().as_posix()
print(f"{PARITION_BY_DIR = }")

# Multi-column partitionBy("country", "gender") — both columns are
# low-cardinality, so the nested `country=/gender=` directory count stays
# small. This is the well-behaved multi-partition case.
(
    df.write.format("csv")
            .mode("overwrite")
            .option("header", True)
            .partitionBy("country", "gender")
            .save(PARITION_BY_DIR)
)


In [0]:
dbutils.fs.ls('/Volumes/workspace/default/write_ops_data_output/parition_by_country_gender/country=INDIA/gender=m/')

### Non-ideal Partitioning — high-cardinality columns

Partitioning on a high-cardinality column (e.g. `id`, `user_id`) defeats the purpose of `partitionBy()`.

[!WARNING]
When every value is unique, Spark creates **one directory per value** — N distinct ids → N partition directories. The result is a flood of tiny files, no pruning benefit, and slower reads/writes.

[!NOTE]
This is exactly the scenario where **bucketing** is the better choice (see the next section) — a fixed number of bucket files keeps the layout manageable regardless of cardinality.

In [0]:
from pathlib import Path
PARITION_BY_DIR = (Path(WRITE_OPS_OUTPUT_DIR) / "parition_by_id" ).resolve().as_posix()
print(f"{PARITION_BY_DIR = }")

# partitionBy("id") — the HIGH-cardinality anti-pattern: one directory per
# unique id (15 dirs for 15 rows). This produces a flood of tiny files with
# no pruning benefit. bucketBy() / liquid clustering is the right tool for
# high-cardinality keys (see cells 19-20).
(
    df.write.format("csv")
            .mode("overwrite")
            .option("header", True)
            .option("path", PARITION_BY_DIR)
            .partitionBy("id")
            .save()
)


In [0]:
%sh 

ls -l /Volumes/workspace/default/write_ops_data_output/parition_by_id/

In [0]:
df.show()

# Bucketing — `bucketBy()`

Decomposes data into a **fixed number of buckets** by hashing one or more columns. Each row lands in exactly one bucket; the bucket count stays constant regardless of column cardinality, which avoids the small-files problem that plagues high-cardinality `partitionBy()`.

[!NOTE]
Primary purpose: **eliminate expensive shuffles** during subsequent `JOIN` or `GROUP BY` operations on large datasets. Rows with the same key always land in the same bucket, so matching keys are already co-located.

### Syntax

`bucketBy()` requires a **persistent metastore table** (`saveAsTable`) — it cannot be used with direct file writes like `.write.parquet("path")`.

```python
# Bucket data into 10 buckets by user_id, pre-sorted within each bucket
df.write \
    .format("parquet") \
    .bucketBy(10, "user_id") \
    .sortBy("registration_date") \
    .saveAsTable("my_bucketed_table")
```

[!TIP]
💡 `.sortBy()` is optional but recommended — it pre-sorts rows within each bucket, speeding up range filters and merge joins downstream.

### How it skips the shuffle

By default, joining two large DataFrames on a key forces a **network shuffle** so matching IDs land on the same executor. With both tables bucketed by that key:

1. Spark knows exactly which bucket holds which IDs.
2. It performs a **bucket-aware join**, matching buckets directly.
3. The shuffle is **bypassed** — a slow, resource-heavy operation becomes a fast local one.

### Crucial rules for the shuffle to be skipped

[!IMPORTANT]
All three conditions must hold, or Spark falls back to a normal shuffle:
- **Same number of buckets** on both tables (e.g. both have 10)
- **Same join key** — the join condition must match the bucketed column exactly
- **Persistent tables** on both sides — read back via `spark.table("name")`, not an in-memory DataFrame

### When to use bucketing

| Scenario | Why bucketing helps |
| --- | --- |
| Frequent large joins | A big dimension table (e.g. `users`) joined daily with large fact tables — bucket once, skip the shuffle every time |
| Large `GROUP BY` queries | Repeated aggregations on the same high-cardinality key — buckets pre-group the rows |

### Bucketing vs. partitioning — at a glance

| Aspect | `partitionBy()` | `bucketBy()` |
| --- | --- | --- |
| Output structure | One directory per distinct value | Fixed N bucket files |
| Pruning mechanism | Directory pruning at read time | Hash-based, same bucket = same key |
| File count | Grows with cardinality | Fixed at write time |
| Supported sinks | File path (`save`) or table (`saveAsTable`) | Persistent table only (`saveAsTable`) |
| Best for | Filters / `WHERE` on the column | Joins / `GROUP BY` on the column |

In [0]:
from pathlib import Path
BUCKET_BY_DIR = (Path(WRITE_OPS_OUTPUT_DIR) / "bucket_by_id" ).resolve().as_posix()
print(f"{BUCKET_BY_DIR = }")

# Legacy Hive bucketBy() — kept for reference. It is NOT supported on Delta
# tables, and Unity Catalog managed tables are Delta-only, so this cell
# raises DELTA_OPERATION_NOT_ALLOWED on run. Cell 20 shows the working
# UC/Delta-native equivalent: liquid clustering (CLUSTER BY).
(
    df.write.format("delta")
            .mode("overwrite")
            .bucketBy(3, "id")
            .saveAsTable("bucket_by_id_table")
)


In [0]:
# Liquid clustering — the UC/Delta-native replacement for `bucketBy()`.
#
# `bucketBy()` (cell 19) is a legacy Hive feature that Delta does NOT support.
# On Unity Catalog managed tables (Delta-only), use `CLUSTER BY` instead:
# it co-locates rows with the same key so joins / GROUP BYs on it skip the
# shuffle — exactly the benefit Hive bucketing provided on a legacy metastore.
#
# `CLUSTER BY` is a table property set at creation time via SQL (not a
# DataFrameWriter option), so we create the table with a CTAS statement.

df.createOrReplaceTempView("source_df")

spark.sql("""
    CREATE OR REPLACE TABLE workspace.default.bucket_by_id_table
    CLUSTER BY (id)
    AS SELECT * FROM source_df
""")

# Read it back and verify
print("Table properties (note 'clusteringColumns'):")
display(spark.sql("SHOW TBLPROPERTIES workspace.default.bucket_by_id_table"))
print("\nTable contents:")
display(spark.table("workspace.default.bucket_by_id_table"))